# Verbal introspection — GPT-2 XL single-token concept set (builder)

**Question.** The injected-thought protocol (`data/experiments/verbal-introspection.json`) scores the
rank of the concept as the *next token* after the teacher-forced prefill. Under the GPT-2 tokenizer
44 of its 101 concepts are several tokens when written without a leading space, which is how they
follow the open quote of the `default` prefill (`... The thought is about "`). Which concept set
keeps every concept a single GPT-2 XL token, so that no trial has to be dropped or scored by a prefix?

**Answer.** This notebook writes `data/experiments/verbal-introspection-gpt2.json`: the same
`intro_prompt` and `prefills`, and 101 concepts that are each one GPT-2 XL token both with and
without a leading space. The 57 original concepts that already pass are kept in their original
order; 44 concrete nouns replace the 44 that fail, chosen round-robin over 12 topic groups.

## Method

* **Tokenizer.** `openai-community/gpt2-xl` (the GPT-2 BPE vocabulary shared by all GPT-2 sizes).
  Only the tokenizer is loaded; no model is run.
* **Single-token rule.** A word passes if `encode(word)` and `encode(" " + word)` are each exactly
  one token. The first form is the next token after the open quote of either prefill; the second is
  the next token for prefills without a quote (e.g. `... is about the word`). Requiring both lets
  either prefill be used with the same concepts. `InjectDirection` needs one `token_id` per concept,
  and the protocol scores the rank of that same token.
* **Kept originals.** Every original concept that passes is kept, in its original order, so results
  remain comparable with the paper's set on the overlap.
* **Replacements.** Candidates are common concrete nouns grouped by topic (food, animals, nature,
  materials, objects, vehicles, buildings, household, body, clothing, people, culture). Candidates
  that fail the rule, duplicate an original concept, or occur as a whole word in the intro prompt
  or the prefills are dropped. The rest are taken round-robin over the groups, in listed order,
  until there are 101 concepts. No model output was used to choose them.

## Setup

In [1]:
import hashlib
import json
import re
from itertools import chain, zip_longest
from pathlib import Path

import pandas as pd
from transformers import AutoTokenizer

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "experiments").is_dir())
SOURCE = REPO / "data/experiments/verbal-introspection.json"
OUTPUT = REPO / "data/experiments/verbal-introspection-gpt2.json"
MODEL_ID = "openai-community/gpt2-xl"

source = json.loads(SOURCE.read_text())
tok = AutoTokenizer.from_pretrained(MODEL_ID)
N_CONCEPTS = len(source["concepts"])
print(N_CONCEPTS, "original concepts; tokenizer vocab", tok.vocab_size)

101 original concepts; tokenizer vocab 50257


In [2]:
def pieces(text):
    return tok.convert_ids_to_tokens(tok.encode(text))


def single_token(word):
    return len(tok.encode(word)) == 1 and len(tok.encode(" " + word)) == 1


prompt_text = " ".join([*(m["content"] for m in source["intro_prompt"]), *source["prefills"].values()])


def in_prompt(word):
    return re.search(rf"(?<!\w){re.escape(word)}(?!\w)", prompt_text, re.I) is not None

## Original concepts

In [3]:
orig = pd.DataFrame(source["concepts"])
orig["no_space"] = orig.surface.map(pieces)
orig["with_space"] = orig.surface.map(lambda w: pieces(" " + w))
orig["single"] = orig.surface.map(single_token)
print(orig.single.sum(), "pass;", (~orig.single).sum(), "fail")
assert orig.single.equals(orig.with_space.str.len().eq(1) & orig.no_space.str.len().eq(1))
assert orig.with_space.str.len().eq(1).all()  # the failures are all no-space splits
print(orig.loc[~orig.single, ["surface", "no_space"]].to_string(index=False))

57 pass; 44 fail
  surface      no_space
lightning [light, ning]
   corner    [cor, ner]
   cinema  [c, inem, a]
     tomb      [t, omb]
   toilet    [to, ilet]
    ocean     [o, cean]
    humor     [hum, or]
   narrow    [n, arrow]
    truck     [t, ruck]
     sock      [s, ock]
     pier      [p, ier]
     dawn      [d, awn]
     ease      [e, ase]
    thumb     [th, umb]
  voltage   [volt, age]
  prophet   [prop, het]
    pasta     [past, a]
      gym       [gy, m]
 painting  [pain, ting]
    medal     [med, al]
     tape      [t, ape]
     epic      [ep, ic]
   garden    [g, arden]
     wool      [w, ool]
    skull     [sk, ull]
   beetle  [be, et, le]
  tension   [t, ension]
    judge     [jud, ge]
   valley    [val, ley]
  diamond   [d, iamond]
      pig       [p, ig]
    curve     [cur, ve]
    lever     [le, ver]
     soap      [so, ap]
   muscle    [mus, cle]
    decay     [dec, ay]
  squeeze [squ, ee, ze]
  formula   [form, ula]
     desk      [des, k]
    tired     [t, ired]

## Replacement candidates

In [4]:
CANDIDATES = {
    "food": "apple bread wine beer egg rice cake",
    "animals": "horse cat bird bear wolf bee fish",
    "nature": "tree flower leaf forest river lake hill rain ice fire cloud moon sun star",
    "materials": "gold silver iron glass stone rock sand wood paper",
    "objects": "book clock phone camera key map flag ball coin sword knife shield gun",
    "vehicles": "car train bike boat",
    "buildings": "bridge tower castle church school hospital prison bank market",
    "household": "door window wall chair table bed",
    "body": "heart blood bone brain eye mouth finger hand foot tail wing",
    "clothing": "shirt coat dress",
    "people": "king doctor baby mother",
    "culture": "song music money game",
}
taken = set(orig.surface)
rows = []
for group, words in CANDIDATES.items():
    for word in words.split():
        reason = ("duplicate" if word in taken else "in prompt" if in_prompt(word)
                  else "multi-token" if not single_token(word) else "")
        rows.append({"group": group, "word": word, "dropped": reason})
cand = pd.DataFrame(rows)
assert not cand.word.duplicated().any()
print(len(cand), "candidates;", cand.dropped.eq("").sum(), "eligible")
cand[cand.dropped.ne("")]

91 candidates; 91 eligible


,group,word,dropped


In [5]:
eligible = [cand[cand.group.eq(g) & cand.dropped.eq("")].word.tolist() for g in CANDIDATES]
round_robin = [w for w in chain.from_iterable(zip_longest(*eligible)) if w]
n_new = N_CONCEPTS - int(orig.single.sum())
added = round_robin[:n_new]
assert len(added) == n_new
cand["added"] = cand.word.isin(added)
cand[cand.added].groupby("group", sort=False).word.agg(["count", ", ".join])

,count,join
group,,
food,4,"apple, bread, wine, beer"
animals,4,"horse, cat, bird, bear"
nature,4,"tree, flower, leaf, forest"
materials,4,"gold, silver, iron, glass"
objects,4,"book, clock, phone, camera"
vehicles,4,"car, train, bike, boat"
buildings,4,"bridge, tower, castle, church"
household,4,"door, window, wall, chair"
body,3,"heart, blood, bone"


## Write the dataset

In [6]:
concepts = [{"name": w, "surface": w, "origin": "original"} for w in orig[orig.single].surface]
concepts += [{"name": w, "surface": w, "origin": "added"} for w in added]
surfaces = [c["surface"] for c in concepts]
assert len(surfaces) == N_CONCEPTS == len(set(surfaces))
assert all(single_token(w) and not in_prompt(w) for w in surfaces)
ids = [tok.encode(" " + w)[0] for w in surfaces]
assert len(set(ids)) == N_CONCEPTS

out = {
    "intro_prompt": source["intro_prompt"],
    "prefills": source["prefills"],
    "concepts": concepts,
    "metadata": {
        "version": 1,
        "source": "data/experiments/verbal-introspection.json",
        "source_sha256": hashlib.sha256(SOURCE.read_bytes()).hexdigest(),
        "tokenizer": MODEL_ID,
        "rule": ("each surface is exactly one token both without and with a leading space; originals "
                 "that pass are kept in order, the rest replaced by concrete nouns taken round-robin "
                 "over topic groups; no surface occurs as a whole word in intro_prompt or prefills"),
        "n_original": int(orig.single.sum()),
        "n_added": len(added),
        "builder": "notebooks/gpt2-xl/verbal_introspection/verbal_introspection_gpt2_build.ipynb",
    },
}
OUTPUT.write_text(json.dumps(out, indent=1, ensure_ascii=False) + "\n")
print(OUTPUT.relative_to(REPO), len(concepts), "concepts")
print("added:", ", ".join(added))

data/experiments/verbal-introspection-gpt2.json 101 concepts
added: apple, horse, tree, gold, book, car, bridge, door, heart, shirt, king, song, bread, cat, flower, silver, clock, train, tower, window, blood, coat, doctor, music, wine, bird, leaf, iron, phone, bike, castle, wall, bone, dress, baby, money, beer, bear, forest, glass, camera, boat, church, chair


## Conclusions

The new set keeps the prompt and both prefills unchanged and has 101 concepts, each a single
GPT-2 XL token with and without a leading space, with 101 distinct token ids. The 57 kept originals
allow a direct comparison with the paper's set on the overlap, which should be reported separately
from the 44 added nouns (`origin`). The added words are concrete and frequent, while several dropped
originals are rarer (`voltage`, `beetle`, `prophet`). Frequency can affect how easily the model emits
a word at baseline, so compare the strength-0 control ranks of the two `origin` groups before pooling
them. Single-token coverage is specific to the GPT-2 tokenizer; rebuild the set for another model.